# 06 — Decorators with Arguments

## Learning Goal

This notebook teaches **arguments passed to the decorator itself**, not function arguments such as `*args` and `**kwargs`.

Previously, we learned:

```python
@decorator
def greet():
    ...
```

Now we want configurable decorators such as:

```python
@repeat(3)
def greet():
    print("Hello")
```

Here:

- `repeat` is the **decorator factory**
- `3` is an **argument to the decorator**
- `greet` is the **function being decorated**
- the decorator ultimately returns the wrapper

The key new idea is the **three-level structure** required by a configurable decorator.

# 1. Introduction

A normal decorator has fixed behavior.

For example:

```python
def uppercase(func):
    def wrapper():
        print("HELLO")
    return wrapper
```

But what if we want to control that behavior?

For example:

```python
@repeat(3)
def greet():
    print("Hello")
```

Output:

```text
Hello
Hello
Hello
```

Or:

```python
@repeat(5)
def greet():
    print("Hello")
```

Output:

```text
Hello
Hello
Hello
Hello
Hello
```

The number `3` or `5` is supplied to the **decorator**, so the decorator needs to become configurable.

# 2. Review: Basic Decorators

This is a short review of File 5.

A basic decorator looks like this:

In [ ]:
def decorator(func):

    def wrapper():
        print("Before")
        func()
        print("After")

    return wrapper

Then:

```python
@decorator
def greet():
    print("Hello")
```

is equivalent to:

```python
greet = decorator(greet)
```

This is only a review. The new topic is how to pass configuration **to the decorator itself**.

# 3. Why Do Decorators Need Arguments?

Consider a fixed decorator:

In [ ]:
def repeat(func):

    def wrapper():
        for _ in range(3):
            func()

    return wrapper

This always repeats the function three times.

We want to be able to choose the number:

```python
@repeat(3)
def greet():
    print("Hello")
```

and:

```python
@repeat(5)
def welcome():
    print("Welcome")
```

The decorator therefore needs to be **configurable**.

# 4. Difference Between the Two Types of Arguments

This distinction is very important.

## Arguments to the decorator

```python
@repeat(3)
def greet():
    ...
```

Here:

```python
3
```

belongs to the **decorator**.

## Arguments to the decorated function

```python
@decorator
def greet(name):
    ...

greet("Komal")
```

Here:

```python
"Komal"
```

belongs to the **function**.

Visualize the difference:

```text
@repeat(3)
    ↑
Decorator argument

greet("Komal")
       ↑
Function argument
```

These are two different stages of the process.

# 5. The Three-Level Structure

This is the most important new concept in this notebook.

A normal decorator has two important levels:

```python
def decorator(func):
    def wrapper():
        func()
    return wrapper
```

A decorator with arguments needs another function level:

```python
def decorator(argument):

    def actual_decorator(func):

        def wrapper():
            func()

        return wrapper

    return actual_decorator
```

There are now three layers:

```text
decorator(argument)
        ↓
actual_decorator(func)
        ↓
wrapper()
```

Each layer has a different responsibility:

| Layer | Receives |
|---|---|
| Outer function | decorator configuration |
| Middle function | function being decorated |
| Inner function | executes the decorated behavior |

# 6. Building a Decorator Factory

A **decorator factory** is a function that receives configuration and returns a decorator.

Build one step by step.

In [ ]:
def repeat(times):
    ...

In [ ]:
def repeat(times):

    def decorator(func):
        ...

    return decorator

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                func()

        return wrapper

    return decorator

Now the factory can be used with `@repeat(3)`:

In [ ]:
@repeat(3)
def greet():
    print("Hello")


greet()

Expected output:

```text
Hello
Hello
Hello
```

# 7. Basic Decorator with One Argument

Let's build the same example while keeping the three responsibilities visible.

### Level 1 — configuration

```python
def repeat(times):
```

`times` stores the decorator configuration.

### Level 2 — decorated function

```python
def decorator(func):
```

`func` is the function that will be decorated.

### Level 3 — wrapper

```python
def wrapper():
```

The wrapper performs the configured behavior.

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                func()

        return wrapper

    return decorator


@repeat(3)
def greet():
    print("Hello")


greet()

Output:

```text
Hello
Hello
Hello
```

The value `3` is remembered by the inner functions because it belongs to the decorator factory's configuration.

# 8. Using `@decorator(argument)` Syntax

Consider:

```python
@repeat(3)
def greet():
    print("Hello")
```

The important point is that `repeat(3)` is evaluated first.

Conceptually:

```python
decorator = repeat(3)
greet = decorator(greet)
```

This is different from:

```python
@repeat
def greet():
    ...
```

In the second form, `repeat` itself directly receives the function.

In the first form, `repeat(3)` first produces a decorator.

# 9. How `@decorator(argument)` Works

Break the process into three steps.

Given:

```python
@repeat(3)
def greet():
    print("Hello")
```

### Step 1 — call the outer function

```python
repeat(3)
```

This returns the actual decorator.

### Step 2 — pass the function to that decorator

Conceptually:

```python
decorator(greet)
```

### Step 3 — the decorator returns the wrapper

The final result is equivalent to:

```python
greet = repeat(3)(greet)
```

A useful mental model is:

```text
repeat(3)
   ↓
returns decorator
   ↓
decorator(greet)
   ↓
returns wrapper
   ↓
greet now refers to wrapper
```

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                func()

        return wrapper

    return decorator


@repeat(3)
def greet():
    print("Hello")


greet()

# 10. Decorators with Multiple Arguments

Decorator factories can receive multiple configuration values.

For example:

```python
def repeat(times, message):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                print(message)
                func()

        return wrapper

    return decorator
```

In [ ]:
def repeat(times, message):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                print(message)
                func()

        return wrapper

    return decorator


@repeat(3, "Calling function")
def greet():
    print("Hello")


greet()

Output:

```text
Calling function
Hello
Calling function
Hello
Calling function
Hello
```

Keyword arguments can also configure the decorator:

```python
@repeat(times=3, message="Running")
def greet():
    print("Hello")
```

Keyword arguments were already covered in the Functions section, so they are not reteached here.

In [ ]:
@repeat(times=3, message="Running")
def welcome():
    print("Welcome")


welcome()

# 11. Using Conditions Inside Configurable Decorators

Decorator arguments can control whether and how the decorated function runs.

For example:

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper():

            if times <= 0:
                return

            for _ in range(times):
                func()

        return wrapper

    return decorator


@repeat(3)
def greet():
    print("Hello")


greet()

The decorator configuration determines how many times the function executes.

Another example is a configurable condition:

In [ ]:
def run_if(condition):

    def decorator(func):

        def wrapper():

            if condition:
                return func()

        return wrapper

    return decorator


@run_if(True)
def greet():
    print("Hello")


greet()

Here the value passed to `run_if()` controls whether the decorated function runs.

# 12. Decorators That Control Behavior

Configurable decorators can provide reusable behavior with different settings.

## Message configuration

```python
def announce(message):

    def decorator(func):

        def wrapper():
            print(message)
            return func()

        return wrapper

    return decorator
```

In [ ]:
def announce(message):

    def decorator(func):

        def wrapper():
            print(message)
            return func()

        return wrapper

    return decorator


@announce("Starting function...")
def greet():
    print("Hello")


greet()

## Prefix configuration

```python
def add_prefix(prefix):

    def decorator(func):

        def wrapper():
            print(prefix, end="")
            return func()

        return wrapper

    return decorator
```

In [ ]:
def add_prefix(prefix):

    def decorator(func):

        def wrapper():
            print(prefix, end="")
            return func()

        return wrapper

    return decorator


@add_prefix("Message: ")
def greet():
    print("Hello")


greet()

The decorator is now reusable while the message or prefix can be changed for each use.

# 13. Practical Decorator Examples

## Example 1 — Repeat execution

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper():
            for _ in range(times):
                func()

        return wrapper

    return decorator


@repeat(3)
def say_hello():
    print("Hello")


say_hello()

## Example 2 — Logging message

In [ ]:
def log_message(message):

    def decorator(func):

        def wrapper():
            print(message)
            return func()

        return wrapper

    return decorator


@log_message("Starting greeting...")
def greet():
    print("Hello")


greet()

## Example 3 — Permission/configuration check

This is a simple learning example showing how decorator configuration and function arguments can be different.

In [ ]:
def require_role(role):

    def decorator(func):

        def wrapper(user_role):
            if user_role == role:
                return func(user_role)

            print("Access denied")

        return wrapper

    return decorator


@require_role("admin")
def dashboard(role):
    print("Welcome to dashboard")


dashboard("admin")
dashboard("user")

Here:

```text
"admin"
    ↓
argument to the decorator

user_role
    ↓
argument passed to the decorated function
```

This is exactly the distinction introduced in Section 4.

# 14. Using `*args` and `**kwargs` in the Wrapper

> **Reference only — do not reteach them here.**

`*args` and `**kwargs` were already covered in:

```text
01_Functions_and_Scope/03_Args_and_Kwargs.ipynb
```

They are useful here because they allow a wrapper to work with functions having different signatures.

For example:

In [ ]:
def repeat(times):

    def decorator(func):

        def wrapper(*args, **kwargs):
            result = None

            for _ in range(times):
                result = func(*args, **kwargs)

            return result

        return wrapper

    return decorator


@repeat(3)
def greet(name):
    print("Hello", name)


greet("Komal")

The important point for this notebook is only the connection:

```python
wrapper(*args, **kwargs)
```

allows the wrapper to pass the decorated function's arguments through.

Do not treat this section as a second lesson on `*args` and `**kwargs`.

# 15. Common Mistakes

## Mistake 1 — Forgetting the extra function level

A decorator with arguments needs the outer factory, the decorator, and the wrapper.

A structure such as:

```python
def repeat(times):
    def wrapper():
        ...
    return wrapper
```

does not correctly receive the decorated function.

The middle `decorator(func)` level is required.

## Mistake 2 — Confusing decorator arguments with function arguments

Given:

```python
@repeat(3)
def greet(name):
    ...
```

remember:

```text
3       → decorator configuration
name    → function parameter
```

## Mistake 3 — Forgetting to return the decorator

Wrong:

```python
def repeat(times):

    def decorator(func):
        ...

    # missing:
    # return decorator
```

Correct:

```python
def repeat(times):

    def decorator(func):
        ...

    return decorator
```

The outer function must return the actual decorator.

## Mistake 4 — Calling the function too early

Keep the reference-vs-call distinction clear.

```python
func
```

refers to the function.

```python
func()
```

calls the function.

This distinction from the earlier Functions material still applies inside decorators.

## Mistake 5 — Forgetting that `@repeat(3)` first calls `repeat`

```python
@repeat(3)
```

is not simply:

```python
@repeat
```

The parentheses mean that the outer function is called first:

```python
repeat(3)
```

and that call must produce a decorator.

# 16. Summary

## Basic decorator

```python
@decorator
def function():
    ...
```

Conceptually:

```python
function = decorator(function)
```

## Decorator with arguments

```python
@decorator(argument)
def function():
    ...
```

The process is:

```text
argument
   ↓
decorator(argument)
   ↓
returns decorator
   ↓
decorator(function)
   ↓
returns wrapper
```

The three-level structure is:

```python
def decorator_factory(configuration):

    def decorator(func):

        def wrapper(*args, **kwargs):
            # behavior
            return func(*args, **kwargs)

        return wrapper

    return decorator
```

### What this notebook teaches

```text
✓ Decorator configuration
✓ Decorator factories
✓ @decorator(argument)
✓ Multiple decorator arguments
✓ Configurable decorator behavior
✓ Three-level decorator structure
✓ How Python evaluates @decorator(argument)
✓ Practical configurable decorators
```

### Intentionally deferred

```text
✗ Reteaching *args/**kwargs
✗ Reteaching function arguments
✗ Closures in depth
✗ functools.wraps
✗ Class decorators
✗ Advanced decorator patterns
```

The key mental model to remember is:

```text
@repeat(3)
def greet():
    ...

        ↓

greet = repeat(3)(greet)
```

That expression captures the essential mechanism of a decorator factory.